# Notebook 15 (Revised) — Read-only evaluation-protocol preflight

**Status:** Completed: read-only evaluation-protocol preflight executed and verified. No terminal data accessed, predictions generated, or scores calculated. This is a **design review**, not an evaluation. No SOLETE HDF5, held-out targets, model predictions, retraining, metrics from the terminal period, or writes are permitted.

**Immutable upstream milestones:** selection lock `c6a9162`; read-only refit preflight Notebook 13; frozen Notebook 14 (`fcf3c6c`); three fitted LightGBM provenance records committed at `565d431`. Model `.txt` weights remain local, deliberately outside Git. The companion `scripts/verify_all_three_final_refits.py` verifies their bytes and native LightGBM structure **without predicting or scoring**.

| Horizon | Frozen candidate | Feature family | Development training rows | Predictor columns |
|---:|---|---|---:|---:|
| 15 min | LightGBM C1 | F1 | 104,817 | 133 |
| 30 min | LightGBM C1 | F2 | 104,814 | 138 |
| 60 min | LightGBM C1 | F2 | 104,808 | 138 |

**Scientific boundary:** June–August 2019 was inspected in the *historical* research, although excluded from revised development CV. Any later use is a **retrospective, historically exposed evaluation**, not an untouched independent test. New protected observations or a separately designed external study are required for stronger independent evidence.

**Execution:** Restart kernel → Run All in the existing repository `.venv`. This notebook only reads the frozen lock and committed model **metadata**, calls the already-vetted read-only model-verification script, and computes *synthetic timestamp geometry*. It does not open an HDF5 file. It does not persist anything other than ordinary Jupyter execution outputs if you choose to save the notebook.

## 1 — Locate the repository and apply fail-closed safety guards

The three flags below are hard assertions, not switches for launching evaluation. This notebook contains no prediction or scoring routine.

In [1]:
from __future__ import annotations
import hashlib
import json
import os
from pathlib import Path
import subprocess
import sys

import pandas as pd
from IPython.display import display

RUN_TRAINING = False
RUN_TERMINAL_EVALUATION = False
RUN_EXTERNAL_EVALUATION = False
assert not (RUN_TRAINING or RUN_TERMINAL_EVALUATION or RUN_EXTERNAL_EVALUATION)

LOCK_COMMIT = "c6a9162"
REFIT_INVENTORY_COMMIT = "565d431"
LOCK_REL = "results/development_only/revised_model_selection_lock_v1.json"
MODEL_REL = "results/final_model_refits/revised_lightgbm_v1"
VERIFY_SCRIPT_REL = "scripts/verify_all_three_final_refits.py"

candidates = ([Path(os.environ["SOLETE_PROJECT_ROOT"]).expanduser()]
              if os.environ.get("SOLETE_PROJECT_ROOT") else [])
candidates += [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((p.resolve() for p in candidates
                     if (p / ".git").exists()
                     and (p / LOCK_REL).is_file()
                     and (p / VERIFY_SCRIPT_REL).is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run from the verified project checkout or set SOLETE_PROJECT_ROOT")

# Jupyter's working directory and Python import search path are independent.
# When the kernel starts from notebooks/ (or another directory), Python may
# not see the repository's src/ package unless the checkout root is inserted.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
assert (PROJECT_ROOT / "src" / "forecasting_protocol.py").is_file(), (
    "Frozen forecasting source not found under the discovered repository root"
)

def git(*args, check=True):
    return subprocess.run(["git", *args], cwd=PROJECT_ROOT,
                          check=check, capture_output=True)

assert git("merge-base", "--is-ancestor", LOCK_COMMIT, "HEAD", check=False).returncode == 0
assert git("merge-base", "--is-ancestor", REFIT_INVENTORY_COMMIT, "HEAD", check=False).returncode == 0
print("REPOSITORY & NO-EVALUATION SAFETY GUARDS: PASSED")
print("HEAD:", git("rev-parse", "--short", "HEAD").stdout.decode().strip())

REPOSITORY & NO-EVALUATION SAFETY GUARDS: PASSED
HEAD: 565d431


## 2 — Freeze the exact committed lock and provenance JSON

Compare local metadata **byte for byte** with the Git snapshots. Neither a changed selection rule nor an edited model hash is acceptable. The lock's historical preparation-time `PROPOSED_LOCK_NOT_YET_COMMITTED` field is kept unchanged because commit `c6a9162` is the effective freeze event.

In [2]:
lock_bytes = (PROJECT_ROOT / LOCK_REL).read_bytes()
assert lock_bytes == git("show", f"{LOCK_COMMIT}:{LOCK_REL}").stdout
lock = json.loads(lock_bytes)
assert lock["freeze_state"] == "PROPOSED_LOCK_NOT_YET_COMMITTED"
assert lock["archived_audit"]["archived_fold_scores"] == 1665
assert lock["evaluation_policy"]["terminal_access_authorized"] is False
assert lock["evaluation_policy"]["can_claim_fresh_independent_test"] is False
assert lock["refit_protocol"]["do_not_fit_or_score_terminal_data"] is True
lock_sha = hashlib.sha256(lock_bytes).hexdigest()

selected = {int(c["horizon_minutes"]): c for c in lock["selected_candidates"]}
assert set(selected) == {15, 30, 60}
expected_families = {15: "F1", 30: "F2", 60: "F2"}
expected_rows = {15: 104817, 30: 104814, 60: 104808}
expected_columns = {15: 133, 30: 138, 60: 138}
expected_sha = {
    15: "aeadd57863612e732c6b98972c2be80b60037dec0769331aab6e1e4c92d17bdf",
    30: "e841a2aa4c2ef5a19ba3c8c9ec708eea5e878c1d2c29ea625234d259eecf6eef",
    60: "aac4d7e3c1c84495f494f783c930ce0917d35e2441356220afffb4eb68c0f023",
}

inventory = []
for h in (15, 30, 60):
    s = selected[h]
    assert s["model"] == "LightGBM" and s["candidate_id"] == "LightGBM_C1"
    assert s["feature_family"] == expected_families[h]
    assert s["configuration"] == {"learning_rate": 0.03, "n_estimators": 300, "num_leaves": 15}
    assert s["refit"]["training_rows_expected"] == expected_rows[h]
    stem = f"lightgbm_c1_h{h}_{expected_families[h]}"
    rel = f"{MODEL_REL}/{stem}.json"
    b = (PROJECT_ROOT / rel).read_bytes()
    assert b == git("show", f"{REFIT_INVENTORY_COMMIT}:{rel}").stdout, f"h{h}: committed model metadata changed"
    r = json.loads(b)
    ident = r["training_identity"]
    assert ident["lock_sha256"] == lock_sha
    assert ident["lock_commit"] == LOCK_COMMIT
    assert ident["training_rows"] == expected_rows[h]
    assert ident["feature_columns"] == expected_columns[h]
    assert ident["feature_family"] == expected_families[h]
    assert ident["configuration"] == s["configuration"]
    assert r["model_sha256"] == expected_sha[h]
    assert r["model_filename"] == stem + ".txt"
    assert r["fitted_boosting_iterations"] == 300
    assert r["terminal_evaluated"] is False and r["predictions_generated"] is False
    inventory.append({
        "horizon_min": h, "model": s["candidate_id"],
        "features": expected_families[h], "columns": expected_columns[h],
        "training_rows": expected_rows[h], "boosting_rounds": 300,
        "cv_mean_mae_kw": s["cv_selection"]["unweighted_fold_mean_mae_kw"],
        "model_sha256": expected_sha[h],
    })

inventory_table = pd.DataFrame(inventory)
display(inventory_table)
print("LOCK & COMMITTED MODEL PROVENANCE: PASSED")
print("No training or terminal data accessed")

,horizon_min,model,features,columns,training_rows,boosting_rounds,cv_mean_mae_kw,model_sha256
0,15,LightGBM_C1,F1,133,104817,300,0.177400,aeadd57863612e732c6b98972c2be80b60037dec076933...
1,30,LightGBM_C1,F2,138,104814,300,0.219329,e841a2aa4c2ef5a19ba3c8c9ec708eea5e878c1d2c29ea...
2,60,LightGBM_C1,F2,138,104808,300,0.280951,aac4d7e3c1c84495f494f783c930ce0917d35e24413562...


LOCK & COMMITTED MODEL PROVENANCE: PASSED
No training or terminal data accessed


## 3 — Independent verification of the actual local fitted weights

The previously committed verifier checks local LightGBM model-file SHA-256, model native structure (300 boosting iterations; 133/138 columns), source lock, and dependency pins. It **does not** read HDF5, fit, predict, score, or write. This is an intentionally strong local gate: a fresh checkout **without backed-up `.txt` files** must fail here rather than pretending that metadata alone constitutes deployable models.

In [3]:
audit = subprocess.run(
    [sys.executable, VERIFY_SCRIPT_REL], cwd=PROJECT_ROOT,
    check=True, capture_output=True, text=True,
)
print(audit.stdout)
assert "ALL THREE FINAL LIGHTGBM REFITS: VERIFIED" in audit.stdout
assert "NO HDF5 READS | NO TRAINING | NO PREDICTIONS | NO SCORING | NO FILE WRITES" in audit.stdout
for h in (15, 30, 60):
    assert f"{h:>2} min: VERIFIED" in audit.stdout
print("ACTUAL LOCAL MODEL FILES: READ-ONLY AUDIT PASSED")

15 min: VERIFIED — F1 — 104,817 development rows — 133 features — 300 trees — sha256=aeadd57863612e732c6b98972c2be80b60037dec0769331aab6e1e4c92d17bdf
30 min: VERIFIED — F2 — 104,814 development rows — 138 features — 300 trees — sha256=e841a2aa4c2ef5a19ba3c8c9ec708eea5e878c1d2c29ea625234d259eecf6eef
60 min: VERIFIED — F2 — 104,808 development rows — 138 features — 300 trees — sha256=aac4d7e3c1c84495f494f783c930ce0917d35e2441356220afffb4eb68c0f023
ALL THREE FINAL LIGHTGBM REFITS: VERIFIED — source, lock, model bytes, metadata, native structure
NO HDF5 READS | NO TRAINING | NO PREDICTIONS | NO SCORING | NO FILE WRITES
NOTE: The original refit notebooks recomputed each development-input SHA. This stand-alone audit verifies the recorded hashes' consistency without rereading SOLETE.

ACTUAL LOCAL MODEL FILES: READ-ONLY AUDIT PASSED


## 4 — Future retrospective-period geometry, using synthetic timestamps only

**No test/HDF5 measurements are loaded.** The reference window is specified by the historical protocol solely to predeclare what a *future separately authorized* retrospective run would score. The first forecast issued for a horizon may occur shortly before the June 1 valid-time boundary; that is intentional. Avoid the error of filtering by issue time and inadvertently dropping valid-time rows.

- Score forecasts indexed by **target-valid time**, not issue time.
- All three horizons cover `2019-06-01 00:00:00 UTC ≤ valid_time < 2019-09-01 00:00:00 UTC`.
- The 5-minute valid-time grid contains **26,496 values per horizon**, before any future data-integrity checks.
- A future evaluator must confirm that every input is available at the forecast issue time. The 60-minute gap is an embargo on *training labels*, not a ban on later observed issue-time covariates.

In [4]:
from src.forecasting_protocol import (
    FORECAST_HORIZONS, MODEL_AVAILABILITY_GAP,
    TEST_START, TEST_END, TRAIN_START,
)
assert tuple(FORECAST_HORIZONS) == (15, 30, 60)
assert MODEL_AVAILABILITY_GAP == pd.Timedelta(minutes=60)
assert TEST_START == pd.Timestamp("2019-06-01T00:00:00Z")
assert TEST_END == pd.Timestamp("2019-09-01T00:00:00Z")

# Synthetic calendar only; never the HDF5 timestamp/measurement array.
valid_grid = pd.date_range(TEST_START, TEST_END, freq="5min", inclusive="left", tz="UTC")
assert len(valid_grid) == 26496
assert valid_grid[-1] == pd.Timestamp("2019-08-31 23:55:00Z")
retro_geometry = []
for h in (15, 30, 60):
    issues = valid_grid - pd.Timedelta(minutes=h)
    assert len(issues) == len(valid_grid)
    assert issues[-1] + pd.Timedelta(minutes=h) == valid_grid[-1]
    assert issues[0] > pd.Timestamp("2019-05-31 22:55:00Z")
    assert (issues < valid_grid).all()
    # Both baselines below are time-available by each issue, without reading values.
    assert ((valid_grid - pd.Timedelta(days=1)) <= issues).all()
    retro_geometry.append({
        "horizon_minutes": h,
        "first_issue_utc": str(issues[0]),
        "first_valid_utc": str(valid_grid[0]),
        "last_valid_utc": str(valid_grid[-1]),
        "potential_scored_rows": len(valid_grid),
    })
geometry_table = pd.DataFrame(retro_geometry)
display(geometry_table)
print("RETROSPECTIVE SCORING GRID DESIGN: PASSED (timestamps only; no measurements)")

,horizon_minutes,first_issue_utc,first_valid_utc,last_valid_utc,potential_scored_rows
0,15,2019-05-31 23:45:00+00:00,2019-06-01 00:00:00+00:00,2019-08-31 23:55:00+00:00,26496
1,30,2019-05-31 23:30:00+00:00,2019-06-01 00:00:00+00:00,2019-08-31 23:55:00+00:00,26496
2,60,2019-05-31 23:00:00+00:00,2019-06-01 00:00:00+00:00,2019-08-31 23:55:00+00:00,26496


RETROSPECTIVE SCORING GRID DESIGN: PASSED (timestamps only; no measurements)


## 5 — Predeclare descriptive scoring rules (not executed)

The following rules are a **draft evaluation design**, **not approval to inspect the historical test**. Prior to executing any later scoring script, the user must separately approve this design and archive a specific evaluation plan. No tuning, refitting, recalibration, or winner switching is allowed after inspecting retrospective outcomes.

| Topic | Proposed rule |
|---|---|
| Primary scoring unit | One forecast per 5-minute *valid-time* timestamp and horizon |
| Target | `P_Solar[kW]`, units kW |
| Scores | MAE (kW) and RMSE (kW), each computed across a fixed, verified eligible row set; report exact `n` |
| Model selection | Already frozen. LightGBM F1 15m, F2 30m and F2 60m; no terminal tuning |
| Fair comparison | Common, explicitly audited set of valid times for any model/baseline comparison within a horizon |
| Baseline 1 | Issue-time persistence: `P(t)` as prediction for `P(t+h)` |
| Baseline 2 | Same valid-time yesterday: `P(t+h−24h)`, a past observation when `h≤60m` |
| Missing values / alignment | Fail closed; report missing or invalid rows, do not silently fill, clip, or choose a favorable scoring subset |
| Negative/out-of-range predictions | Preserve raw predictions; no post-hoc clipping or recalibration without a separately frozen rule |
| Uncertainty | No significance or independent-confirmation claim from previously inspected June–August 2019; any uncertainty analysis must be prespecified separately |
| Reporting | Always mark retrospective SOLETE window *historically exposed*. Development-CV scores are descriptive provenance, not unbiased test estimates |
| Reproducibility | Record row-index digest, model SHA, code commit, data/input provenance, feature order, missingness, prediction digest, and exact metrics |

Future protected/new data should have a **separate eligibility, timestamp-alignment, station-compatibility, and model-applicability protocol**. The PVOD feature schema does not automatically match SOLETE's frozen 133/138-column models. External transfer must not be advertised as direct plug-and-play.

In [5]:
# Purely declarative; contains no SOLETE targets, model predictions or scoring code.
evaluation_design = {
    "status": "PROPOSED_EVALUATION_DESIGN_NOT_AUTHORIZED",
    "model_selection_commit": LOCK_COMMIT,
    "model_inventory_commit": REFIT_INVENTORY_COMMIT,
    "historical_terminal_previously_inspected": True,
    "may_claim_fresh_independent_test": False,
    "evaluation_executed": False,
    "terminal_access_authorized": False,
    "training_or_tuning_authorized": False,
    "horizons_minutes": [15, 30, 60],
    "target": "P_Solar[kW]",
    "metrics": ["MAE_kW", "RMSE_kW"],
    "retrospective_valid_start_utc": "2019-06-01T00:00:00Z",
    "retrospective_valid_end_utc_exclusive": "2019-09-01T00:00:00Z",
    "reference_rows_per_horizon": 26496,
    "baselines": ["PV_at_forecast_issue", "PV_at_valid_time_minus_24h"],
    "feature_family_by_horizon": {15: "F1", 30: "F2", 60: "F2"},
    "unapproved_model_retraining_or_switching": False,
    "missing_data_policy": "fail_closed_and_report_not_silently_dropped",
}
assert not evaluation_design["evaluation_executed"]
assert not evaluation_design["terminal_access_authorized"]
assert not evaluation_design["training_or_tuning_authorized"]
assert evaluation_design["historical_terminal_previously_inspected"]
assert evaluation_design["reference_rows_per_horizon"] == len(valid_grid)
assert evaluation_design["feature_family_by_horizon"] == expected_families
print("PROPOSED EVALUATION DESIGN (READ-ONLY — NO APPROVAL IMPLIED)")
print(json.dumps(evaluation_design, indent=2))

PROPOSED EVALUATION DESIGN (READ-ONLY — NO APPROVAL IMPLIED)
{
  "status": "PROPOSED_EVALUATION_DESIGN_NOT_AUTHORIZED",
  "model_selection_commit": "c6a9162",
  "model_inventory_commit": "565d431",
  "historical_terminal_previously_inspected": true,
  "may_claim_fresh_independent_test": false,
  "evaluation_executed": false,
  "terminal_access_authorized": false,
  "training_or_tuning_authorized": false,
  "horizons_minutes": [
    15,
    30,
    60
  ],
  "target": "P_Solar[kW]",
  "metrics": [
    "MAE_kW",
    "RMSE_kW"
  ],
  "retrospective_valid_start_utc": "2019-06-01T00:00:00Z",
  "retrospective_valid_end_utc_exclusive": "2019-09-01T00:00:00Z",
  "reference_rows_per_horizon": 26496,
  "baselines": [
    "PV_at_forecast_issue",
    "PV_at_valid_time_minus_24h"
  ],
  "feature_family_by_horizon": {
    "15": "F1",
    "30": "F2",
    "60": "F2"
  },
  "unapproved_model_retraining_or_switching": false,
  "missing_data_policy": "fail_closed_and_report_not_silently_dropped"
}


## 6 — Acceptance and next authorization boundary

Passing this notebook means only that **the model inventory and proposed evaluation design are internally consistent**. It does **not** mean that terminal scores are now approved or that a new independent test exists. Do not edit the frozen models, selection JSON, source fingerprints or training notebooks.

Before the next stage, explicitly choose and document whether to perform:

1. **Historically exposed SOLETE retrospective assessment** — useful as a transparent descriptive audit, not a fresh independent performance claim; or
2. **A genuinely new protected-data / external validation study** — requires compatibility checks and a separate station/data protocol before any inference.

Do not run both implicitly or substitute one for the other. Separately, matched F3→F4 Panchang analysis from development CV addresses representation questions, **not** the selected primary model's terminal scoring.

In [6]:
assert not (RUN_TRAINING or RUN_TERMINAL_EVALUATION or RUN_EXTERNAL_EVALUATION)
assert set(inventory_table["horizon_min"].tolist()) == {15, 30, 60}
assert inventory_table["training_rows"].tolist() == [104817, 104814, 104808]
assert inventory_table["columns"].tolist() == [133, 138, 138]
assert geometry_table["potential_scored_rows"].tolist() == [26496] * 3
assert evaluation_design["evaluation_executed"] is False
assert evaluation_design["terminal_access_authorized"] is False
print("NOTEBOOK 15 EVALUATION-PROTOCOL PREFLIGHT: PASSED")
print("MODEL INVENTORY: 3/3 verified against committed JSON and local weights")
print("RETROSPECTIVE VALID-TIME REFERENCE: 26,496 timestamps per horizon (synthetic calendar)")
print("NO HDF5 / TERMINAL TARGET READS | NO FITS | NO PREDICTIONS | NO SCORES | NO WRITES")
print("RETROSPECTIVE / EXTERNAL EVALUATION: BOTH NOT AUTHORIZED")

NOTEBOOK 15 EVALUATION-PROTOCOL PREFLIGHT: PASSED
MODEL INVENTORY: 3/3 verified against committed JSON and local weights
RETROSPECTIVE VALID-TIME REFERENCE: 26,496 timestamps per horizon (synthetic calendar)
NO HDF5 / TERMINAL TARGET READS | NO FITS | NO PREDICTIONS | NO SCORES | NO WRITES
RETROSPECTIVE / EXTERNAL EVALUATION: BOTH NOT AUTHORIZED
